In [ ]:
!pwd

2101.94s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
/kaggle/working


In [ ]:
!nvidia-smi

In [17]:
!ps aux | grep -E "vllm|api_server" | grep -v grep

root         225 10.6  6.1 8780100 2018280 ?     Sl   12:25   0:48 /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-VL-2B-Instruct --served-model-name arc-local --host 127.0.0.1 --port 1234 --dtype half --max-model-len 8192 --gpu-memory-utilization 0.90


In [18]:
import requests

r = requests.get(
    "http://127.0.0.1:1234/v1/models",
    timeout=10,
)

print(r.status_code)
print(r.json())

200
{'object': 'list', 'data': [{'id': 'arc-local', 'object': 'model', 'created': 1788957192, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3-VL-2B-Instruct', 'parent': None, 'max_model_len': 8192, 'permission': [{'id': 'modelperm-a938da1ad348222f', 'object': 'model_permission', 'created': 1788957192, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [19]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="local",
)

response = client.chat.completions.create(
    model="arc-local",
    messages=[
        {
            "role": "user",
            "content": "Reply exactly with: LOCAL MODEL WORKS",
        }
    ],
    temperature=0,
    max_tokens=50,
)

print(response.choices[0].message.content)

LOCAL MODEL WORKS


In [21]:
from PIL import Image, ImageDraw
import io
import base64

image = Image.new(
    "RGB",
    (256, 256),
    "white",
)

draw = ImageDraw.Draw(image)

draw.rectangle(
    (50, 50, 200, 200),
    fill="red",
)

buffer = io.BytesIO()
image.save(buffer, format="PNG")

encoded = base64.b64encode(
    buffer.getvalue()
).decode("ascii")

image_url = (
    "data:image/png;base64,"
    + encoded
)

response = client.chat.completions.create(
    model="arc-local",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What shape and color do you see?",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                    },
                },
            ],
        }
    ],
    temperature=0,
    max_tokens=100,
)

print(response.choices[0].message.content)

Based on the image provided, I can see the following:

-   **Shape:** The object is a square.
-   **Color:** The color is red.

The image is a simple, solid red square.


In [22]:
import os

os.environ["ARC_LOCAL_BASE_URL"] = (
    "http://127.0.0.1:1234/v1"
)

os.environ["ARC_LOCAL_MODEL"] = (
    "arc-local"
)